# FL Anomaly Detection — Adaptive Aggregation
**Run every cell top to bottom. Never skip.**
Cells: 1=Install → 2=Paths → 3=Data → 4=Visualise → 5=Phase1 → 6=Phase2 → 7=Graphs → 8=Table → 9=Save

In [ ]:
# CELL 1 — Install dependencies
import subprocess, sys, torch

for pkg in ['scikit-learn','seaborn','pyyaml','tqdm','requests']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Ready | Device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# CELL 2 — Mount Drive and fix Python paths
import os, sys, shutil

from google.colab import drive
drive.mount('/content/drive')

# Auto-detect where the project lives
CANDIDATES = [
    '/content/fl_anomaly_detection/fl_anomaly_detection',
    '/content/fl_anomaly_detection',
    '/content/drive/MyDrive/fl_anomaly_detection/fl_anomaly_detection',
    '/content/drive/MyDrive/fl_anomaly_detection',
]

def find_root():
    for p in CANDIDATES:
        if os.path.exists(os.path.join(p, 'data', 'download.py')):
            return p
    return None

REPO_ROOT = find_root()

# If not found locally, copy from Drive
if REPO_ROOT is None:
    for drive_p in ['/content/drive/MyDrive/fl_anomaly_detection',
                    '/content/drive/MyDrive/fl_anomaly_detection/fl_anomaly_detection']:
        if os.path.exists(os.path.join(drive_p, 'data', 'download.py')):
            shutil.copytree(drive_p, '/content/fl_project', dirs_exist_ok=True)
            REPO_ROOT = '/content/fl_project'
            break

if REPO_ROOT is None:
    raise RuntimeError(
        'Cannot find project folder.\n'
        'Upload fl_anomaly_detection to Google Drive first.\n'
        'Expected structure: fl_anomaly_detection/data/download.py'
    )

# Add to Python path
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

# Create all required directories
for d in ['results/logs','results/plots','results/checkpoints',
          'data/raw/nsl_kdd','data/processed/nsl_kdd']:
    os.makedirs(os.path.join(REPO_ROOT, d), exist_ok=True)

print(f'✅ REPO_ROOT = {REPO_ROOT}')
print(f'   Files: {sorted(os.listdir(REPO_ROOT))}')

In [ ]:
# CELL 3 — Download and preprocess NSL-KDD
import os, sys, numpy as np

# Re-ensure path (safe to repeat)
for p in sys.path:
    if 'fl_anomaly_detection' in p and os.path.exists(os.path.join(p,'data','download.py')):
        REPO_ROOT = p
        break
os.chdir(REPO_ROOT)

from data.download   import download_nsl_kdd
from data.preprocess import preprocess_nsl_kdd, load_processed

print('Downloading NSL-KDD...')
download_nsl_kdd()

print('Preprocessing...')
X_train, X_test, y_train, y_test, y_bin_train, y_bin_test = preprocess_nsl_kdd()

print(f'\n📊 NSL-KDD loaded:')
print(f'  Train : {X_train.shape}  | Normal={np.sum(y_bin_train==0):,} Attack={np.sum(y_bin_train==1):,}')
print(f'  Test  : {X_test.shape}   | Normal={np.sum(y_bin_test==0):,}  Attack={np.sum(y_bin_test==1):,}')
print(f'  dtype : X={X_train.dtype} y={y_bin_train.dtype}')
print('✅ Data ready')

In [ ]:
# CELL 4 — Visualise IID vs Non-IID data distribution
import matplotlib.pyplot as plt
import numpy as np
from data.partition import get_partitions

NUM = 5
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
CASES  = [('IID','iid',None),('Non-IID α=0.5','non_iid',0.5),('Non-IID α=0.1','non_iid',0.1)]
COLS   = ['#4CAF50','#F44336','#2196F3','#FF9800','#9C27B0']
LABELS = ['Normal','DoS','Probe','R2L','U2R']

for ax, (title, strat, alpha) in zip(axes, CASES):
    parts = get_partitions(X_train, y_train, NUM, strat, alpha or 0.5)
    data  = np.zeros((NUM, 5))
    for i,(_, yc) in enumerate(parts):
        for c in range(5):
            data[i,c] = np.sum(np.array(yc)==c)
    pct = data / np.maximum(data.sum(1,keepdims=True), 1) * 100
    bot = np.zeros(NUM)
    for c in range(5):
        ax.bar(range(NUM), pct[:,c], bottom=bot, color=COLS[c], label=LABELS[c], alpha=0.85)
        bot += pct[:,c]
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Client'); ax.set_ylabel('Class %')
    ax.set_xticks(range(NUM)); ax.set_xticklabels([f'C{i}' for i in range(NUM)])

axes[2].legend(loc='upper right', fontsize=8)
plt.suptitle('Data Distribution Across Clients', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/plots/0_data_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graph 0 saved → results/plots/0_data_distribution.png')

In [ ]:
# CELL 5 — Phase 1: FedAvg vs Adaptive (5 clients, 50 rounds)
# Quick validation — ~10-15 min on T4
import copy, yaml, torch
from data.partition       import get_partitions
from data.preprocess      import load_processed
from federation.simulation import FLSimulation

with open('configs/base_config.yaml') as f:
    BASE = yaml.safe_load(f)

BASE['federation']['num_clients']  = 5
BASE['federation']['num_rounds']   = 50
BASE['experiment']['device']       = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE['results'] = {
    'log_dir':             'results/logs',
    'plot_dir':            'results/plots',
    'checkpoint_dir':      'results/checkpoints',
    'save_every_n_rounds': 25
}

X_train, X_test, _, _, y_bin_train, y_bin_test = load_processed('nsl_kdd')
phase1_results = {}

for strategy in ['fedavg', 'adaptive']:
    for (part, alpha) in [('iid', None), ('non_iid', 0.1)]:
        label    = 'IID' if part == 'iid' else 'NonIID_a0.1'
        exp_name = f'{strategy}_{label}'
        print(f'\n{"="*45}\nRunning: {exp_name}\n{"="*45}')

        cfg = copy.deepcopy(BASE)
        cfg['experiment']['name']              = exp_name
        cfg['aggregation']['strategy']         = strategy
        cfg['partitioning']['strategy']        = part
        cfg['partitioning']['dirichlet_alpha'] = float(alpha or 0.5)

        parts = get_partitions(X_train, y_bin_train, 5, part, float(alpha or 0.5))
        try:
            sim = FLSimulation(cfg)
            sim.setup(parts, X_test, y_bin_test)
            res = sim.run()
            phase1_results[exp_name] = res
            m   = res['final_metrics']
            print(f'  ✅ AUC={m["auc_roc"]:.4f} F1={m["f1"]:.4f} DR={m["dr"]:.4f} FAR={m["far"]:.4f}')
        except Exception as e:
            import traceback
            print(f'  ❌ {exp_name}: {e}')
            traceback.print_exc()

print(f'\n✅ Phase 1 done: {len(phase1_results)}/4')

In [ ]:
# CELL 6 — Phase 2: Full sweep (4 strategies x 4 conditions = 16 experiments)
# Runtime: ~30-60 min on T4. Run before sleep.
import copy, yaml, torch, json, os
import numpy as np
from data.partition        import get_partitions
from data.preprocess       import load_processed
from federation.simulation import FLSimulation

with open('configs/base_config.yaml') as f:
    BASE = yaml.safe_load(f)

BASE['federation']['num_clients']  = 10
BASE['federation']['num_rounds']   = 100
BASE['experiment']['device']       = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE['results'] = {
    'log_dir':             'results/logs',
    'plot_dir':            'results/plots',
    'checkpoint_dir':      'results/checkpoints',
    'save_every_n_rounds': 10
}

X_train, X_test, _, _, y_bin_train, y_bin_test = load_processed('nsl_kdd')

STRATEGIES  = ['fedavg', 'fedprox', 'scaffold', 'adaptive']
CONDITIONS  = [('iid',None), ('non_iid',1.0), ('non_iid',0.5), ('non_iid',0.1)]
TOTAL       = len(STRATEGIES) * len(CONDITIONS)
phase2_results = {}
done = 0

for strategy in STRATEGIES:
    for (part, alpha) in CONDITIONS:
        done     += 1
        cond      = 'IID' if part == 'iid' else f'alpha={alpha}'
        exp_name  = f'{strategy}_{cond}'
        print(f'\n{"="*55}\n[{done}/{TOTAL}] {exp_name}\n{"="*55}')

        cfg = copy.deepcopy(BASE)
        cfg['experiment']['name']              = exp_name
        cfg['aggregation']['strategy']         = strategy
        cfg['partitioning']['strategy']        = part
        cfg['partitioning']['dirichlet_alpha'] = float(alpha if alpha is not None else 0.5)

        parts = get_partitions(
            X=X_train, y=y_bin_train, num_clients=10,
            strategy=part, alpha=float(alpha if alpha is not None else 0.5))

        try:
            sim = FLSimulation(cfg)
            sim.setup(parts, X_test, y_bin_test)
            res = sim.run()
            phase2_results[exp_name] = res
            m   = res['final_metrics']
            cv  = res.get('convergence_round', 100)
            print(f'  ✅ AUC={m["auc_roc"]:.4f} F1={m["f1"]:.4f} DR={m["dr"]:.4f} FAR={m["far"]:.4f} Conv={cv}')
        except Exception as e:
            import traceback
            print(f'  ❌ FAILED: {e}')
            traceback.print_exc()
            continue

# Save JSON
with open('results/phase2_all_results.json','w') as f:
    json.dump({k: v['final_metrics'] for k,v in phase2_results.items()}, f, indent=2)

print(f'\n✅ Phase 2 done: {len(phase2_results)}/{TOTAL}')
print(f'💾 Saved → results/phase2_all_results.json')

In [ ]:
# CELL 7 — Generate all 6 research graphs
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np, os

matplotlib.rcParams.update({
    'figure.dpi':150,'font.family':'DejaVu Sans',
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.3
})
os.makedirs('results/plots', exist_ok=True)

METHODS   = ['FedAvg','FedProx','SCAFFOLD','Adaptive FL']
SMAP      = {'FedAvg':'fedavg','FedProx':'fedprox','SCAFFOLD':'scaffold','Adaptive FL':'adaptive'}
CONDS     = ['IID','alpha=1.0','alpha=0.5','alpha=0.1']
CLABELS   = ['IID','α=1.0','α=0.5','α=0.1']
COLORS    = ['#2196F3','#F44336','#4CAF50','#9C27B0']

def gm(method, cond, metric):
    key = f'{SMAP[method]}_{cond}'
    return phase2_results.get(key,{}).get('final_metrics',{}).get(metric, 0.0)

def gc(method, cond):
    key = f'{SMAP[method]}_{cond}'
    return phase2_results.get(key,{}).get('convergence_round', 100)

def grv(method, cond, metric):
    key = f'{SMAP[method]}_{cond}'
    rds  = [r['round']           for r in phase2_results.get(key,{}).get('rounds',[])]
    vals = [r['metrics'][metric] for r in phase2_results.get(key,{}).get('rounds',[])]
    return rds, vals

# ── Graph 1: Convergence curves ──────────────────────────────
fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax,(cond,ls) in zip(axes,[('IID','-'),('alpha=0.1','--')]):
    for i,m in enumerate(METHODS):
        rds,vals = grv(m,cond,'auc_roc')
        if rds:
            ax.plot(rds,vals,color=COLORS[i],linestyle=ls,label=m,linewidth=2)
    ax.set_title(f'{"IID" if cond=="IID" else "Non-IID α=0.1"}',fontweight='bold')
    ax.set_xlabel('Round'); ax.set_ylabel('AUC-ROC')
    ax.set_ylim(0.5,1.0); ax.legend(fontsize=9)
plt.suptitle('Convergence Curves',fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig('results/plots/1_convergence.png',bbox_inches='tight')
plt.show(); print('✅ Graph 1')

# ── Graph 2: F1 bar chart ─────────────────────────────────────
fig,ax = plt.subplots(figsize=(12,5))
x,w = np.arange(len(CONDS)),0.18
for i,m in enumerate(METHODS):
    ax.bar(x+i*w,[gm(m,c,'f1') for c in CONDS],w,label=m,color=COLORS[i],alpha=0.85)
ax.set_xticks(x+w*1.5); ax.set_xticklabels(CLABELS)
ax.set_ylabel('Macro F1'); ax.set_title('F1 Score Comparison',fontweight='bold')
ax.set_ylim(0,1.05); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('results/plots/2_f1_bars.png',bbox_inches='tight')
plt.show(); print('✅ Graph 2')

# ── Graph 3: AUC Heatmap ──────────────────────────────────────
fig,ax = plt.subplots(figsize=(8,4))
mat = np.array([[gm(m,c,'auc_roc') for c in CONDS] for m in METHODS])
sns.heatmap(mat,annot=True,fmt='.3f',cmap='RdYlGn',
            xticklabels=CLABELS,yticklabels=METHODS,
            vmin=0.5,vmax=1.0,linewidths=0.5,ax=ax,
            cbar_kws={'label':'AUC-ROC'})
ax.set_title('AUC-ROC Heatmap',fontweight='bold')
plt.tight_layout()
plt.savefig('results/plots/3_heatmap.png',bbox_inches='tight')
plt.show(); print('✅ Graph 3')

# ── Graph 4: DR vs FAR scatter ────────────────────────────────
fig,ax = plt.subplots(figsize=(7,6))
markers=['o','s','^','D']
for i,m in enumerate(METHODS):
    for j,c in enumerate(CONDS):
        dr,far = gm(m,c,'dr'), gm(m,c,'far')
        if dr > 0:
            ax.scatter(far,dr,color=COLORS[i],marker=markers[i],
                       s=80+j*25,alpha=0.85,
                       label=m if j==0 else '')
ax.set_xlabel('False Alarm Rate'); ax.set_ylabel('Detection Rate')
ax.set_title('DR vs FAR',fontweight='bold'); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('results/plots/4_dr_far.png',bbox_inches='tight')
plt.show(); print('✅ Graph 4')

# ── Graph 5: Adaptive weight spread over rounds ───────────────
fig,ax = plt.subplots(figsize=(10,5))
for cond,ls,label in [('alpha=0.1','-','α=0.1'),('alpha=0.5','--','α=0.5')]:
    key  = f'adaptive_{cond}'
    rnds = [r['round'] for r in phase2_results.get(key,{}).get('rounds',[])
            if r.get('alpha_weights')]
    stds = [np.std(r['alpha_weights']) for r in phase2_results.get(key,{}).get('rounds',[])
            if r.get('alpha_weights')]
    if rnds:
        ax.plot(rnds,stds,linestyle=ls,linewidth=2,color='#9C27B0',label=f'Adaptive {label}')
ax.set_xlabel('Round'); ax.set_ylabel('Std of α_i weights')
ax.set_title('Adaptive Weight Spread Over Training',fontweight='bold')
ax.legend(); plt.tight_layout()
plt.savefig('results/plots/5_adaptive_weights.png',bbox_inches='tight')
plt.show(); print('✅ Graph 5')

# ── Graph 6: Convergence rounds bar ──────────────────────────
fig,ax = plt.subplots(figsize=(12,5))
x,w = np.arange(len(CONDS)),0.18
for i,m in enumerate(METHODS):
    ax.bar(x+i*w,[gc(m,c) for c in CONDS],w,label=m,color=COLORS[i],alpha=0.85)
ax.set_xticks(x+w*1.5); ax.set_xticklabels(CLABELS)
ax.set_ylabel('Rounds to Converge'); ax.set_title('Convergence Speed (fewer=better)',fontweight='bold')
ax.legend(fontsize=9); plt.tight_layout()
plt.savefig('results/plots/6_conv_rounds.png',bbox_inches='tight')
plt.show(); print('✅ Graph 6')

print('\n✅ All 6 graphs saved → results/plots/')

In [ ]:
# CELL 8 — Print full results table and save CSV
import pandas as pd

rows = []
for exp, res in sorted(phase2_results.items()):
    m  = res.get('final_metrics', {})
    cv = res.get('convergence_round', '?')
    rows.append({
        'Experiment':  exp,
        'AUC-ROC':     round(m.get('auc_roc',0), 4),
        'F1':          round(m.get('f1',0),       4),
        'DR':          round(m.get('dr',0),        4),
        'FAR':         round(m.get('far',0),       4),
        'Conv.Round':  cv
    })

df = pd.DataFrame(rows)
print('='*75)
print('FULL RESULTS TABLE')
print('='*75)
print(df.to_string(index=False))

df.to_csv('results/phase2_results.csv', index=False)
print('\n💾 Saved → results/phase2_results.csv')

print('\n' + '='*75)
print('KEY FINDINGS')
print('='*75)
for label, suffix in [('IID', '_IID'), ('Non-IID α=0.1', '_alpha=0.1')]:
    sub = df[df['Experiment'].str.endswith(suffix)]
    if len(sub):
        best = sub.loc[sub['AUC-ROC'].idxmax()]
        print(f'{label:15} → Best: {best.Experiment:<30} AUC={best["AUC-ROC"]} F1={best["F1"]}')

In [ ]:
# CELL 9 — Save all results to Google Drive (run after every session)
import shutil, os
from datetime import datetime

ts    = datetime.now().strftime('%Y%m%d_%H%M')
dst   = f'/content/drive/MyDrive/fl_results_{ts}'
shutil.copytree('results', dst)

plots = len([f for f in os.listdir(dst+'/plots') if f.endswith('.png')])
logs  = len(os.listdir(dst+'/logs'))
ckpts = len(os.listdir(dst+'/checkpoints'))
print(f'✅ Saved to Google Drive: fl_results_{ts}')
print(f'   Plots:       {plots}')
print(f'   Logs:        {logs}')
print(f'   Checkpoints: {ckpts}')